# TTC Subway Delay Analysis
Monthly delay statistics — 2014 to present.

In [1]:
from pathlib import Path

import polars as pl
import altair as alt

In [2]:
RAW = Path("../data/raw/ttc-subway-delay-data/ttc-subway-delay-may-december-2017.xlsx")

df_raw = pl.read_excel(RAW)
df_raw.head(5)

Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
date,str,str,str,str,i64,i64,str,str,i64
2017-05-01,"""00:18""","""Monday""","""KIPLING STATION (ENTER""","""TUSC""",0,0,"""W""","""BD""",5251
2017-05-01,"""00:58""","""Monday""","""ISLINGTON STATION""","""MUSC""",0,0,"""W""","""BD""",5182
2017-05-01,"""01:07""","""Monday""","""UNION STATION (DOWNSVI""","""MUPAA""",3,7,"""N""","""YU""",5811
2017-05-01,"""01:18""","""Monday""","""MCCOWAN STATION""","""MRTO""",4,9,"""S""","""SRT""",3006
2017-05-01,"""01:42""","""Monday""","""CASTLE FRANK STATION""","""MUO""",7,11,"""W""","""BD""",5296


In [3]:
MONTHLY_PATH = Path("../data/prep/ttc-subway-delay-data/monthly.csv")

monthly = pl.read_csv(MONTHLY_PATH)
monthly

year_month,month_label,delays,total_delay,max_delay,major_delays
i64,str,i64,i64,i64,i64
201401,"""Jan 2014""",1783,4587,292,21
201402,"""Feb 2014""",1881,4465,788,14
201403,"""Mar 2014""",1879,3588,167,13
201404,"""Apr 2014""",1616,3569,198,23
201405,"""May 2014""",1599,3057,98,18
…,…,…,…,…,…
202509,"""Sep 2025""",1811,4179,149,18
202510,"""Oct 2025""",2027,5150,76,41
202511,"""Nov 2025""",2208,6187,480,32


In [4]:
 with pl.Config(tbl_rows=-1):
      display(monthly.with_columns(
          (pl.col("year_month") // 100).alias("year")
      ).group_by("year").len().sort("year"))

year,len
i64,u32
2014,12
2015,12
2016,12
2017,5
2018,1
2019,1
2020,1
2021,1
2022,12


In [5]:
month_order = monthly["month_label"].to_list()

base = alt.Chart(monthly).encode(
    x=alt.X("month_label:N", sort=month_order, title="Month"),
)

chart_total = base.transform_calculate(
    total_delay_days="datum.total_delay / 1440"
).mark_bar(color="#4C72B0").encode(
    y=alt.Y("total_delay_days:Q", title="Total Delay (days)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("total_delay_days:Q", title="Total Delay (days)", format=".2f"),
    ],
).properties(title="Total Delay per Month", width=1250, height=180)

chart_delays = base.mark_bar(color="#55A868").encode(
    y=alt.Y("delays:Q", title="Number of Delays"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("delays:Q", title="Delays"),
    ],
).properties(title="Number of Delays per Month", width=1250, height=180)

chart_major = base.mark_bar(color="#DD8452").encode(
    y=alt.Y("major_delays:Q", title="Major Delays (>= 20 min)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("major_delays:Q", title="Major Delays"),
    ],
).properties(title="Major Delays per Month (>= 20 min)", width=1250, height=180)

alt.vconcat(chart_total, chart_delays, chart_major).properties(spacing=10)

alt.VConcatChart(...)